In [57]:
import os
import glob
import pandas as pd
import torch
import random
import numpy as np

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Configuration of data and features
DATA_ROOT = '/kaggle/input/datasets/rohitkumarkhajekar/ml-dataset/ML_2026A2_Hurricane_dataset'
VAR_INFO = [
    ('Rainf_tavg',    'rain'),
    ('Wind_f_inst',   'wind_spd'),
    ('Psurf_f_inst',  'pressure'),
    ('SWdown_f_tavg', 'sw_down'),
    ('LWdown_f_tavg', 'lw_down'),
    ('Wind_dir_p1000','wind_dir'),
]
VAR_NAMES = ['rain','wind_spd','pressure','sw_down','lw_down','wind_dir_sin','wind_dir_cos']
HOUR_COLS = [f'data_hour_{h}' for h in range(0,24,3)]

# Dimensionalities of Input Data
INPUT_DAYS = 14
FORECAST_DAYS = 7
RAINFALL_SCALE = 86400.0

# Splitting data for hurricane forecasting into a time-series problem
TRAIN_END = pd.Timestamp('2023-12-31')
VAL_START = pd.Timestamp('2024-10-01')
VAL_END = pd.Timestamp('2024-12-10')
TEST_START = pd.Timestamp('2024-12-11')

# Labelling for output prediction
VAR_IDX = np.array([0,1,5,6,2], dtype=np.int64)  # rain, wind_spd, sin, cos, pressure

# Hyperparameters settings
HIDDEN_DIMS  = [512,256,128,64]
DROPOUT_RATE = 0.3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 128
MAX_EPOCHS = 30
PATIENCE = 10
LR_PATIENCE = 5
LR_FACTOR = 0.5

SAVE_DIR = "processed_splits"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Loading the data monthly and merging the variables
def load_month(year, month):
    dfs = []
    for folder, name in VAR_INFO:
        files = glob.glob(f"{DATA_ROOT}/{folder}/*{year}_{month:02d}*.csv")

        if len(files) == 0:
            raise FileNotFoundError(f"No file for {folder} {year}-{month:02d}")

        p = files[0]
        df = pd.read_csv(p)
        df[name] = df[HOUR_COLS].mean(axis=1)
        dfs.append(df[['year','month','day','latitude','longitude', name]])
    m = dfs[0]
    for d in dfs[1:]:
        m = m.merge(d, on=['year','month','day','latitude','longitude'])
    m['date'] = pd.to_datetime(m[['year','month','day']])
    return m


# Preprocessing the data by removing missing values and converting the wind direction to sine and cosine components to avoid discontinuity
def preprocess(df):
    df['wind_dir'] = df['wind_dir'].fillna(0)

    rad = np.deg2rad(df['wind_dir'])
    df['wind_dir_sin'] = np.sin(rad)
    df['wind_dir_cos'] = np.cos(rad)
    df = df.drop(columns=['wind_dir'])

    # Rainfall scaling into logarithmic
    df['rain'] = np.log1p(df['rain'] * RAIN_SCALE)

    # Featuring into numerical variables before training the neural network
    for c in ['wind_spd','pressure','sw_down','lw_down']:
        mu = df[c].mean()
        sd = df[c].std()
        if sd < 1e-6:
            sd = 1.0
        df[c] = (df[c] - mu) / sd

    # Limiting extreme numerical values
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].clip(-5, 5)

    return df
    
# Generating time-series input sequences and forecasting values
def make_sequences(df):
    X, y, dates = [], [], []
    for (_, _), g in df.groupby(['latitude','longitude']):
        g = g.sort_values('date')
        vals = g[VAR_NAMES].values
        dts  = g['date'].values
        for i in range(INPUT_DAYS, len(vals) - FORECAST_DAYS):
            X.append(vals[i-INPUT_DAYS:i].flatten())
            y.append(vals[i+FORECAST_DAYS][VAR_IDX])
            dates.append(dts[i+FORECAST_DAYS])
    return np.array(X), np.array(y), np.array(dates)

# Splitting the data into training set, validation set and test set
def split_time(X,y,dates):
    t = np.datetime64(TRAIN_END)
    v0 = np.datetime64(VAL_START)
    v1 = np.datetime64(VAL_END)
    te = np.datetime64(TEST_START)
    tr = dates <= t
    va = (dates >= v0) & (dates <= v1)
    ts = dates >= te
    return (X[tr],y[tr]), (X[va],y[va]), (X[ts],y[ts])

prev_tail = None

# Processing monthly weather data and generate forecasting datasets
for year in [2023, 2024, 2025]:
    for month in range(1, 13):
        try:
            df = load_month(year, month)
        except Exception as e:
            print(f"Error loading {year}-{month:02d}:", e)
            continue

        if prev_tail is not None:
            df = pd.concat([prev_tail, df], ignore_index=True)

        df = preprocess(df)

        X, y, dates = make_sequences(df)
        (Xt, yt), (Xv, yv), (Xs, ys) = split_time(X, y, dates)

        if len(Xt):
            np.savez_compressed(f"{SAVE_DIR}/train_{year}_{month:02d}.npz", X=Xt, y=yt)

        if len(Xv):
            np.savez_compressed(f"{SAVE_DIR}/val_{year}_{month:02d}.npz", X=Xv, y=yv)

        if len(Xs):
            np.savez_compressed(f"{SAVE_DIR}/test_{year}_{month:02d}.npz", X=Xs, y=ys)

        prev_tail = df.tail(INPUT_DAYS + FORECAST_DAYS)


In [60]:
# Loading input sequences and target labels
class NPZDataset(Dataset):
    def __init__(self, path):
        d = np.load(path)
        self.X = d['X']; self.y = d['y']
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return (torch.tensor(self.X[i], dtype=torch.float32),
                torch.tensor(self.y[i], dtype=torch.float32))

# Multi-Layer Perceptron (MLP) forecasting model
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        layers = []
        dims = [in_dim] + HIDDEN_DIMS + [out_dim]
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i],dims[i+1]), nn.ReLU(), nn.Dropout(DROPOUT_RATE)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

train_files = sorted(glob.glob(f"{SAVE_DIR}/train_*.npz"))
val_files   = sorted(glob.glob(f"{SAVE_DIR}/val_*.npz"))
test_files  = sorted(glob.glob(f"{SAVE_DIR}/test_*.npz"))

# Determing input and output dimensions
sample = np.load(train_files[0])
INPUT_DIM = sample['X'].shape[1]
OUTPUT_DIM = sample['y'].shape[1]

model = MLP(INPUT_DIM, OUTPUT_DIM).to(DEVICE)
opt = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
crit = nn.MSELoss()
sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=LR_PATIENCE, factor=LR_FACTOR)

best_val = float('inf')
wait = 0


In [61]:
# Training the model
for epoch in range(MAX_EPOCHS):

    model.train()
    train_loss = 0

    for file in train_files:
        loader = DataLoader(NPZDataset(file), batch_size=BATCH_SIZE, shuffle=True)

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            if torch.isnan(X_batch).any() or torch.isnan(y_batch).any():
                continue

            preds = model(X_batch)
            loss = crit(preds, y_batch)

            opt.zero_grad()
            loss.backward()
            opt.step()

            train_loss += loss.item()

    train_loss /= len(train_files)

    # Validating
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for file in val_files:
            loader = DataLoader(NPZDataset(file), batch_size=BATCH_SIZE, shuffle=False)

            for X_batch, y_batch in loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                if torch.isnan(X_batch).any() or torch.isnan(y_batch).any():
                    continue

                preds = model(X_batch)
                loss = crit(preds, y_batch)

                val_loss += loss.item()

    val_loss /= len(val_files)

    # Learning Rate Scheduling
    sched.step(val_loss)

    # print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

    # Early Stopping
    if val_loss < best_val:
        best_val = val_loss
        wait = 0
        torch.save(model.state_dict(), "new_model.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping")
            break

Early stopping


In [62]:
# Testing the model
model.load_state_dict(torch.load("new_model.pt", map_location=DEVICE))
model.eval()

print("Test model loaded.")

# Evaluating the model on the test dataset
all_preds = []
all_targets = []

with torch.no_grad():

    for file in test_files:

        loader = DataLoader(
            NPZDataset(file),
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(DEVICE)

            preds = model(X_batch).cpu().numpy()
            targets = y_batch.numpy()

            all_preds.append(preds)
            all_targets.append(targets)

preds = np.vstack(all_preds)
targets = np.vstack(all_targets)

# Computing mse and mae
mse = mean_squared_error(targets, preds)
mae = mean_absolute_error(targets, preds)

print(f"\nTest MSE : {mse:.4f}")
print(f"Test MAE : {mae:.4f}")

# # Creating dataframe containing predictions and ground truth values
df_out = pd.DataFrame({

    "rain_true": targets[:, 0],
    "rain_pred": preds[:, 0],

    "wind_spd_true": targets[:, 1],
    "wind_spd_pred": preds[:, 1],

    "wind_dir_sin_true": targets[:, 2],
    "wind_dir_sin_pred": preds[:, 2],

    "wind_dir_cos_true": targets[:, 3],
    "wind_dir_cos_pred": preds[:, 3],

    "pressure_true": targets[:, 4],
    "pressure_pred": preds[:, 4],
})

CSV_NAME = "Forcasting_results.csv"

df_out.to_csv(CSV_NAME, index=False)

print(f"\nSaved submission file: {CSV_NAME}")
print(df_out.head(4))

Test model loaded.

Test MSE : 0.4332
Test MAE : 0.4803

Saved submission file: Forcasting_results.csv
   rain_true  rain_pred  wind_spd_true  wind_spd_pred  wind_dir_sin_true  \
0        0.0   0.286838       0.069817       0.103708          -0.792411   
1        0.0   0.230693       0.314794       0.042911          -0.994979   
2        0.0   0.243943      -0.017589       0.019778          -0.727453   
3        0.0   0.197068      -0.057885      -0.062556          -0.704015   

   wind_dir_sin_pred  wind_dir_cos_true  wind_dir_cos_pred  pressure_true  \
0          -0.035335          -0.609987           0.121461      -0.088886   
1          -0.076223          -0.100080           0.115484      -0.230329   
2          -0.109102          -0.686157           0.117552      -0.165479   
3          -0.233908          -0.710185           0.119038       0.135918   

   pressure_pred  
0      -0.237862  
1      -0.218247  
2      -0.228249  
3      -0.219015  
